In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
# import missingno as msno
# from plotnine import *
import folium

In [2]:
import requests
# from bs4 import BeautifulSoup
# from selenium import webdriver
import json
from pandas import json_normalize
import time

따릉이 서버에 post 방식으로 요청해서 따릉이 서버가 응답하는 데이터를 받는다.

In [3]:
response = requests.post(
    'https://www.bikeseoul.com/app/station/getStationRealtimeStatus.do',
    data = {
        'stationGrpSeq': 'ALL'
    }
)
# print(response) # <Response [200]>
# print(type(response.text)) # <class str>
print(response.text)

{"stationImgPath":"/nas_link/spb/attachFiles/file_admin/basePath","appUserSessionVO":{"encPwd":null,"usrClsCd":null,"usrDeviceId":null,"usrSeq":null,"voucherEndDttm":null,"voucherSeq":null,"usrType":null,"loginId":null,"usrMpnNo":null,"snsType":null,"mbId":null,"mpnLostYn":null,"lang":"LAG_001","appOsType":null,"orgType":null,"snsId":null,"viewFlg":"list","usrBirthDate":null,"sexCd":null,"rentEncPwd":null,"telecomClsCd":null,"authCiVal":null,"authClsCd":null,"mbTelNo":null,"mbEmailName":null,"mbPostNo":null,"mbAddr1":null,"mbAddr2":null,"parentSexCd":null,"parentBirthDate":null,"parentMpnNo":null,"emailSendAgreeYn":null,"lastLoginDttm":null,"mbWgt":null,"langClsCd":null,"leaveYn":null,"leaveReasonCd":null,"leaveDttm":null,"mbInfoColecAgreeDttm":null,"mpnLostDttm":null,"pagingYn":null,"usrIp":null,"partyVoucherSeq":null,"elecVoucherSeq":null,"requestSeq":null,"usrDeviceType":null,"authDiVal":null,"snsEmail":null,"tCardYn":null,"mCardYn":null,"mainType":null,"regDttm":null,"pageSize":0,"

따릉이 서버가 응답하는 json 형태의 문자열을 파이썬에서 사용하기 위해서 딕셔너리나 리스트 타입으로 변환한다.

In [4]:
# json 모듈의 loads() 메소드를 사용한다.
# bike = json.loads(response.text)
# requests 모듈의 응답 객체에서 json() 메소드를 사용한다.
bike = response.json()
# print(type(bike)) # <class 'dict'>
# print(bike)
print(bike.keys())

dict_keys(['stationImgPath', 'appUserSessionVO', 'loginYn', 'realtimeList', 'sessionId', 'stationVO', 'checkResult'])


딕셔너리 타입으로 변환된 데이터에서 'realtimeList'라는 key에 할당된 정보를 얻어온다. => 따릉이 스테이션 정보

In [5]:
# print(bike.get('realtimeList'))
print(len(bike['realtimeList']))
bike['realtimeList']

2735


[{'stationId': 'ST-4',
  'stationImgFileName': '',
  'stationName': '102. 망원역 1번출구 앞',
  'stationLongitude': '126.91062927',
  'stationLatitude': '37.55564880',
  'rackTotCnt': '15',
  'parkingBikeTotCnt': '0',
  'parkingQRBikeCnt': '2',
  'parkingELECBikeCnt': '0',
  'stationSeCd': 'RAK_002',
  'mode': None},
 {'stationId': 'ST-5',
  'stationImgFileName': '',
  'stationName': '103. 망원역 2번출구 앞',
  'stationLongitude': '126.91083527',
  'stationLatitude': '37.55495071',
  'rackTotCnt': '14',
  'parkingBikeTotCnt': '0',
  'parkingQRBikeCnt': '4',
  'parkingELECBikeCnt': '2',
  'stationSeCd': 'RAK_002',
  'mode': None},
 {'stationId': 'ST-6',
  'stationImgFileName': '',
  'stationName': '104. 합정역 1번출구 앞',
  'stationLongitude': '126.91508484',
  'stationLatitude': '37.55073929',
  'rackTotCnt': '13',
  'parkingBikeTotCnt': '0',
  'parkingQRBikeCnt': '6',
  'parkingELECBikeCnt': '0',
  'stationSeCd': 'RAK_002',
  'mode': None},
 {'stationId': 'ST-7',
  'stationImgFileName': '',
  'stationNam

json_normalize() 함수를 이용해서 따릉이 스테이션 정보를 판다스 데이터프레임으로 만든다.  
json_normalize(딕셔너리, 데이터프레임으로 만들려는 딕셔너리의 key)

In [6]:
bike_df = json_normalize(bike, 'realtimeList')
bike_df

,stationId,stationImgFileName,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt,stationSeCd,mode
0,ST-4,,102. 망원역 1번출구 앞,126.91062927,37.55564880,15,0,2,0,RAK_002,None
1,ST-5,,103. 망원역 2번출구 앞,126.91083527,37.55495071,14,0,4,2,RAK_002,None
2,ST-6,,104. 합정역 1번출구 앞,126.91508484,37.55073929,13,0,6,0,RAK_002,None
3,ST-7,,105. 합정역 5번출구 앞,126.91482544,37.55000687,5,0,0,0,RAK_002,None
4,ST-8,,106. 합정역 7번출구 앞,126.91282654,37.54864502,12,0,3,0,RAK_002,None
...,...,...,...,...,...,...,...,...,...,...,...
2730,ST-3418,,6185.가양나들목,126.84345245,37.57341003,8,0,5,2,RAK_002,None
2731,ST-3415,,6187.마곡119안전센터 맞은편,126.82072449,37.55534744,12,0,2,4,RAK_002,None
2732,ST-3419,,6188.금호아파트,126.86475372,37.55611038,17,0,38,7,RAK_002,None
2733,ST-3424,,6189.데시앙플렉스 지식산업센터,126.84830475,37.56448364,11,0,2,0,RAK_002,None


In [7]:
bike_df.columns

Index(['stationId', 'stationImgFileName', 'stationName', 'stationLongitude',
       'stationLatitude', 'rackTotCnt', 'parkingBikeTotCnt',
       'parkingQRBikeCnt', 'parkingELECBikeCnt', 'stationSeCd', 'mode'],
      dtype='str')

bike_df 데이터프레임에서 아래의 컬럼만 사용하기위해 불필요한 컬럼은 제거한다.

stationId: 대여소 id  
stationName': 대여소 이름  
stationLongitude': 대여소 경도  
stationLatitude': 대여소 위도  
rackTotCnt': 대여소에 주차 가능한 자전거 대수  
parkingBikeTotCnt': 대여소에 주차된 LCD형 따릉이 대수 => 사용하지 않음  
parkingQRBikeCnt': 대여소에 주차된 QR형 따릉이 대수 => 일반 따릉이  
parkingELECBikeCnt': 대여소에 주차된 새싹 따릉이 대수

In [8]:
# bike = bike_df.drop(columns=['stationImgFileName', 'stationSeCd', 'mode'])
bike = bike_df[['stationId', 'stationName', 'stationLongitude', 'stationLatitude', 'rackTotCnt', 'parkingBikeTotCnt','parkingQRBikeCnt', 'parkingELECBikeCnt']]
del bike_df
bike

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt
0,ST-4,102. 망원역 1번출구 앞,126.91062927,37.55564880,15,0,2,0
1,ST-5,103. 망원역 2번출구 앞,126.91083527,37.55495071,14,0,4,2
2,ST-6,104. 합정역 1번출구 앞,126.91508484,37.55073929,13,0,6,0
3,ST-7,105. 합정역 5번출구 앞,126.91482544,37.55000687,5,0,0,0
4,ST-8,106. 합정역 7번출구 앞,126.91282654,37.54864502,12,0,3,0
...,...,...,...,...,...,...,...,...
2730,ST-3418,6185.가양나들목,126.84345245,37.57341003,8,0,5,2
2731,ST-3415,6187.마곡119안전센터 맞은편,126.82072449,37.55534744,12,0,2,4
2732,ST-3419,6188.금호아파트,126.86475372,37.55611038,17,0,38,7
2733,ST-3424,6189.데시앙플렉스 지식산업센터,126.84830475,37.56448364,11,0,2,0


위도, 경도, 전체 자전거 주차 대수를 기억할 파생 변수 등을 고려해서 데이터 타입을 변경한다.

In [9]:
# bike.dtypes
bike.info()

<class 'pandas.DataFrame'>
RangeIndex: 2735 entries, 0 to 2734
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   stationId           2735 non-null   str  
 1   stationName         2735 non-null   str  
 2   stationLongitude    2735 non-null   str  
 3   stationLatitude     2735 non-null   str  
 4   rackTotCnt          2735 non-null   str  
 5   parkingBikeTotCnt   2735 non-null   str  
 6   parkingQRBikeCnt    2735 non-null   str  
 7   parkingELECBikeCnt  2735 non-null   str  
dtypes: str(8)
memory usage: 171.1 KB


In [10]:
bike['stationLongitude'] = bike.stationLongitude.astype(float)
bike['stationLatitude'] = bike.stationLatitude.astype(float)
bike['parkingBikeTotCnt'] = bike.parkingBikeTotCnt.astype(int)
bike['parkingQRBikeCnt'] = bike.parkingQRBikeCnt.astype(int)
bike['parkingELECBikeCnt'] = bike.parkingELECBikeCnt.astype(int)
# 따릉이 스테이션에 현재 주차된 자전거 대수를 기억하는 파생 변수를 만든다.
bike['parking'] = bike['parkingBikeTotCnt'] + bike['parkingQRBikeCnt'] + bike['parkingELECBikeCnt']
bike.head()

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt,parking
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,2,0,2
1,ST-5,103. 망원역 2번출구 앞,126.910835,37.554951,14,0,4,2,6
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,6,0,6
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,0,0,0
4,ST-8,106. 합정역 7번출구 앞,126.912827,37.548645,12,0,3,0,3


In [11]:
bike.to_csv('./data/bike.csv', index=False)
bike.dtypes

stationId                 str
stationName               str
stationLongitude      float64
stationLatitude       float64
rackTotCnt                str
parkingBikeTotCnt       int64
parkingQRBikeCnt        int64
parkingELECBikeCnt      int64
parking                 int64
dtype: object

서울시 전체 따릉이 스테이션 시각화

In [12]:
bike_map = folium.Map(location=[bike.stationLatitude.mean(), bike.stationLongitude.mean()], zoom_start=12)

for i in bike.index[::10]:
    # 따릉이 스테이션 이름앞에 '숫자.'이 있는 경우와 없는 경우를 대비해서 try ~ except를 사용해서 스테이션 이름을 처리한다.
    try:
        stationName = bike.loc[i, 'stationName'].split('.')[1].strip()
    except:
        stationName = bike.loc[i, 'stationName'].split('.')[0].strip()
    # print() 함수에서 '\n'을 사용하면 줄이 변경되지만 folium의 popup은 내부적으로 HTML로 렌더링되기 때문에 '\n'가 무시된다.
    # folium의 popup을 사용할때 줄을 바꾸려면 '\n' 대신에 HTML의 줄바꿈 태그 '<br>'을 사용해야 한다.
    string = '스테이션 이름: {}<br>현재 주차된 자전거 대수: {}대'.format(
        stationName,
        bike.loc[i, 'parking']
    )
    popup = folium.Popup(string, max_width=300)
    
    folium.Marker(
        location=[bike.loc[i, 'stationLatitude'], bike.loc[i, 'stationLongitude']], 
        popup=popup,
    ).add_to(bike_map)

bike_map.save('./output/bike_map.html')
bike_map

GeoCoding

지오코딩: 주소로 위도, 경도 정보 얻기, 역지오코딩: 위도, 경도로 주소 정보 얻기

지오코딩을 사용하기 위해서 라이브러리를 설치하고 import 한다.

GeocoderUnavailable: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=%EC%84%9C%EC%9A%B8%EC%8B%9C+%EC%A2%85%EB%A1%9C%EA%B5%AC+%EA%B4%80%EC%B2%A0%EB%8F%99&format=json&limit=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1091)')))  
위와 같은 에러가 발생되면 아래의 코드를 추가하고 실행한다.

import certifi  
import ssl  
import geopy.geocoders  

ctx = ssl.create_default_context(cafile=certifi.where())  
geopy.geocoders.options.default_ssl_context = ctx

In [13]:
# !pip install geopy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderRateLimited

# GeocoderRateLimited: Non-successful status code 429 에러 방지를 위해서 Nominatim 객체를 함수가 실행될때마다 생성하지 않고 미리 생성한다.
# user_agent 속성값을 다른 사용자와 겹치지 않게 나만의 고유한 이름으로 변경한다.
geocoder = Nominatim(user_agent='my_unique_geocode_id_mkvryu_20260709', timeout=None)

지오코딩(주소를 인수로 넘겨받아 위도, 경도를 리턴하는) 함수를 만든다.

In [14]:
def geocoding(address):
    # Nominatim() 객체가 함수가 실행될때마다 생성된다.
    # geocoder = Nominatim(user_agent='South Korea', timeout=None)
    geo = geocoder.geocode(address)
    return {'위도': geo.latitude, '경도': geo.longitude}

address = geocoding('서울시 종로구 관철동')  
print(address)  
address = geocoding('서울시 강남구 역삼동')  
print(address)

역지오코딩(위도, 경도를 인수로 넘겨받아 주소를 리턴하는) 함수를 만든다.

In [45]:
def geocoding_reverse(lat_lon):
    # Nominatim() 객체가 함수가 실행될때마다 생성된다.
    # geocoder = Nominatim(user_agent='South Korea', timeout=None)
    while True:
        try:
            time.sleep(1.5)
            return geocoder.reverse(lat_lon)
        except GeocoderRateLimited:
            print("서버 차단 감지! 20초간 대기 후 재시도합니다...")
            time.sleep(20)
        except:
            print(f"기타 에러:")
            return None
        finally:
            pass

address = geocoding_reverse('37.5687846,126.9857059')  
print(type(address))  
print(address)  
address = str(address).split(', ')  
print(type(address))  
print(address)  
print(address[-3], address[-4], address[-5])  
print('=' * 100)  

address = geocoding_reverse('37.49988,127.03856')  
address = str(address).split(', ')  
print(address)  
print(address[-3], address[-4], address[-5])

역지오코딩을 통해서 위도와 경도로 얻어진 구(goo)와 동(dong)을 파생 변수로 추가한다.

In [22]:
bike['goo'] = np.nan
bike['dong'] = np.nan
bike['goo'] = bike['goo'].astype(str)
bike['dong'] = bike['dong'].astype(str)
bike

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt,parking,goo,dong
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,2,0,2,NaN,NaN
1,ST-5,103. 망원역 2번출구 앞,126.910835,37.554951,14,0,4,2,6,NaN,NaN
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,6,0,6,NaN,NaN
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,0,0,0,NaN,NaN
4,ST-8,106. 합정역 7번출구 앞,126.912827,37.548645,12,0,3,0,3,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2730,ST-3418,6185.가양나들목,126.843452,37.573410,8,0,5,2,7,NaN,NaN
2731,ST-3415,6187.마곡119안전센터 맞은편,126.820724,37.555347,12,0,2,4,6,NaN,NaN
2732,ST-3419,6188.금호아파트,126.864754,37.556110,17,0,38,7,45,NaN,NaN
2733,ST-3424,6189.데시앙플렉스 지식산업센터,126.848305,37.564484,11,0,2,0,2,NaN,NaN


In [28]:
'''
for i in range(bike.shape[0])[:]:
    lat_lon = f"{bike.loc[i, 'stationLatitude']},{bike.loc[i, 'stationLongitude']}"
    # print(lat_lon)
    address = geocoding_reverse(lat_lon)
    address = str(address).split(', ')
    print(address[-3], address[-4], address[-5])
    
    try:
        bike.loc[i, 'goo'] = address[-4]
    except:
        print('{}번째 인덱스 따릉이 스테이션의 -4번째 주소 없음'.format(i))
        
    try:
        bike.loc[i, 'dong'] = address[-5]
    except:
        print('{}번째 인덱스 따릉이 스테이션의 -5번째 주소 없음'.format(i))
        
    if (i + 1) % 20 == 0:
        print('================', i + 1, '================')
'''
pass

In [65]:
# bike.to_csv('./data/bike_20260709.csv', index=False)
bike = pd.read_csv('./data/bike_20260709.csv') # cp949 어쩌구 에러가 발생되면 encoding='cp949'를 추가한다.
bike

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt,parking,goo,dong
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,2,0,2,마포구,서교동
1,ST-5,103. 망원역 2번출구 앞,126.910835,37.554951,14,0,4,2,6,마포구,망원1동
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,6,0,6,마포구,서교동
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,0,0,0,마포구,서교동
4,ST-8,106. 합정역 7번출구 앞,126.912827,37.548645,12,0,3,0,3,마포구,합정동
...,...,...,...,...,...,...,...,...,...,...,...
2730,ST-3418,6185.가양나들목,126.843452,37.573410,8,0,5,2,7,강서구,가양1동
2731,ST-3415,6187.마곡119안전센터 맞은편,126.820724,37.555347,12,0,2,4,6,강서구,공항동
2732,ST-3419,6188.금호아파트,126.864754,37.556110,17,0,38,7,45,강서구,염창동
2733,ST-3424,6189.데시앙플렉스 지식산업센터,126.848305,37.564484,11,0,2,0,2,강서구,가양1동


추가된 파생 변수에 발생된 문제점을 처리한다.

In [66]:
bike.info()

<class 'pandas.DataFrame'>
RangeIndex: 2735 entries, 0 to 2734
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   stationId           2735 non-null   str    
 1   stationName         2735 non-null   str    
 2   stationLongitude    2735 non-null   float64
 3   stationLatitude     2735 non-null   float64
 4   rackTotCnt          2735 non-null   int64  
 5   parkingBikeTotCnt   2735 non-null   int64  
 6   parkingQRBikeCnt    2735 non-null   int64  
 7   parkingELECBikeCnt  2735 non-null   int64  
 8   parking             2735 non-null   int64  
 9   goo                 2735 non-null   str    
 10  dong                2735 non-null   str    
dtypes: float64(2), int64(5), str(4)
memory usage: 235.2 KB


In [67]:
bike[bike.goo.isnull() | bike.dong.isnull()]

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt,parking,goo,dong


In [68]:
bike[bike.goo.isnull() | bike.dong.isnull()].stationId

Series([], Name: stationId, dtype: str)

In [69]:
errorList = list(bike[bike.goo.isnull() | bike.dong.isnull()].stationId)
print(type(errorList))
print(errorList)

<class 'list'>
[]


역지오코딩 실행 후 'goo'와 'dong'이라는 파생 변수에 NaN이 있을 경우 실행하는 코드

In [70]:
for error in errorList:
    bike_error = bike[bike.stationId == error]
    # print(bike_error)
    lat_lot = f"{bike_error.iloc[0, 5]},{bike_error.iloc[0, 4]}"
    # print(lat_lot)
    address = geocoding_reverse(lat_lon)
    address = str(address).split(', ')
    bike.loc[i, 'goo'] = address[-3]
    bike.loc[i, 'dong'] = address[-4]
    
bike[bike.goo.isnull() | bike.dong.isnull()]

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt,parking,goo,dong


In [166]:
bike.goo.value_counts()

goo
송파구     218
강서구     194
영등포구    172
강남구     172
노원구     148
서초구     143
마포구     120
강동구     117
구로구     115
양천구     106
종로구      97
은평구      96
중랑구      95
중구       91
동대문구     91
광진구      89
성동구      89
용산구      88
서대문구     81
성북구      80
도봉구      72
금천구      71
동작구      67
관악구      66
강북구      57
Name: count, dtype: int64

In [167]:
bike[(bike.goo == '서울특별시') | (bike.goo == '의정부시') | (bike.goo == '하남시')]

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt,parking,goo,dong


In [164]:
address = geocoding_reverse('37.484833,127.149918')
address = str(address).split(', ')
print(address)

['북위례2', '657-12', '위례대로', '학암동', '위례동', '송파구', '서울특별시', '경기도', '13011', '대한민국']


In [165]:
bike.loc[2655, 'goo'] = '송파구'
bike.loc[2655, 'dong'] = '위례동'

In [168]:
bike.to_csv('./data/bike_20260709_ok.csv', index=False)
bike = pd.read_csv('./data/bike_20260709_ok.csv')
bike

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt,parking,goo,dong
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,2,0,2,마포구,서교동
1,ST-5,103. 망원역 2번출구 앞,126.910835,37.554951,14,0,4,2,6,마포구,망원1동
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,6,0,6,마포구,서교동
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,0,0,0,마포구,서교동
4,ST-8,106. 합정역 7번출구 앞,126.912827,37.548645,12,0,3,0,3,마포구,합정동
...,...,...,...,...,...,...,...,...,...,...,...
2730,ST-3418,6185.가양나들목,126.843452,37.573410,8,0,5,2,7,강서구,가양1동
2731,ST-3415,6187.마곡119안전센터 맞은편,126.820724,37.555347,12,0,2,4,6,강서구,공항동
2732,ST-3419,6188.금호아파트,126.864754,37.556110,17,0,38,7,45,강서구,염창동
2733,ST-3424,6189.데시앙플렉스 지식산업센터,126.848305,37.564484,11,0,2,0,2,강서구,가양1동


In [169]:
bike[bike.goo.isnull() | bike.dong.isnull()]

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt,parking,goo,dong


종로구 따릉이 스테이션 위치 시각화

In [170]:
bike.goo.value_counts()

goo
송파구     218
강서구     194
영등포구    172
강남구     172
노원구     148
서초구     143
마포구     120
강동구     117
구로구     115
양천구     106
종로구      97
은평구      96
중랑구      95
중구       91
동대문구     91
광진구      89
성동구      89
용산구      88
서대문구     81
성북구      80
도봉구      72
금천구      71
동작구      67
관악구      66
강북구      57
Name: count, dtype: int64

In [171]:
data = bike[bike.goo == '종로구']

In [173]:
bike_map = folium.Map(location=[data.stationLatitude.mean(), data.stationLongitude.mean()], zoom_start=14)

for i in data.index:
    try:
        stationName = data.loc[i, 'stationName'].split('.')[1].strip()
    except:
        stationName = data.loc[i, 'stationName'].split('.')[0].strip()
    string = '스테이션 이름: {}<br>현재 주차된 자전거 대수: {}대'.format(
        stationName,
        data.loc[i, 'parking']
    )
    popup = folium.Popup(string, max_width=300)
    folium.Marker(
        location=[data.loc[i, 'stationLatitude'], data.loc[i, 'stationLongitude']], 
        popup=popup,
    ).add_to(bike_map)

bike_map.save('./output/bike_map.html')
bike_map

마포구 서교동 따릉이 스테이션 위치 시각화

In [174]:
data = bike[(bike.goo == '마포구') & (bike.dong == '서교동')]
data

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt,parking,goo,dong
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,2,0,2,마포구,서교동
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,6,0,6,마포구,서교동
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,0,0,0,마포구,서교동
5,ST-9,107. 수협 연남동지점,126.918716,37.557846,10,0,7,0,7,마포구,서교동
6,ST-10,108. 서교동 사거리,126.918617,37.552746,12,0,0,0,0,마포구,서교동
8,ST-18,113. 홍대입구역 2번출구 앞,126.923821,37.557438,28,0,32,0,32,마포구,서교동
9,ST-20,114. 홍대입구역 8번출구 앞,126.924423,37.557060,15,0,5,0,5,마포구,서교동
308,ST-2152,493.홍대입구역 6번출구,126.928856,37.556576,7,0,9,0,9,마포구,서교동
1711,ST-2167,3009. 홍대입구역 사거리,126.920235,37.554501,20,0,5,0,5,마포구,서교동
1712,ST-2168,3010.홍대입구역 3번출구,126.925385,37.558296,15,0,16,1,17,마포구,서교동


In [175]:
bike_map = folium.Map(location=[data.stationLatitude.mean(), data.stationLongitude.mean()], zoom_start=15)

for i in data.index:
    string = ' {}<br>일반따릉이: {}<br>새싹따릉이: {}'.format(
        data.loc[i, 'stationName'],
        data.loc[i, 'parkingQRBikeCnt'],
        data.loc[i, 'parkingELECBikeCnt']
    )
    popup = folium.Popup(string, max_width=300)
    folium.Marker(
        location=[data.loc[i, 'stationLatitude'], data.loc[i, 'stationLongitude']], 
        popup=popup,
    ).add_to(bike_map)

bike_map.save('./output/bike_map.html')
bike_map

따릉이 스테이션 이름에 '마포'가 포함된 따릉이 스테이션 위치 시각화

In [176]:
data = bike[bike.stationName.str.find('마포') >= 0]
data

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt,parking,goo,dong
15,ST-23,121. 마포소방서 앞,126.933174,37.549767,15,0,8,4,12,마포구,신수동
35,ST-204,146. 마포역 2번출구 뒤,126.945824,37.539936,12,0,10,0,10,마포구,용강동
36,ST-205,147. 마포역 4번출구 뒤,126.945915,37.539272,5,0,3,7,10,마포구,도화동
40,ST-211,154. 마포구청역,126.905495,37.560909,11,0,16,0,16,마포구,성산1동
67,ST-343,185. 마포 신수공원 앞,126.934296,37.542545,15,0,1,0,1,마포구,신수동
238,ST-83,407. 마포구 육아종합지원센터,126.883675,37.580631,17,0,9,4,13,마포구,상암동
250,ST-87,419. 홈플러스 마포점 앞,126.899429,37.568420,21,0,14,13,27,마포구,성산2동
252,ST-82,421. 마포구청 앞,126.901863,37.565746,8,0,0,1,1,마포구,성산2동
262,ST-1494,432. 마포중앙도서관,126.908203,37.563339,10,0,0,0,0,마포구,성산1동
1707,ST-2163,3005.마포구청 청사내,126.902184,37.566246,10,0,3,0,3,마포구,성산2동


In [177]:
bike_map = folium.Map(location=[data.stationLatitude.mean(), data.stationLongitude.mean()], zoom_start=14)

for i in data.index:
    string = ' {}<br>일반따릉이: {}<br>새싹따릉이: {}'.format(
        data.loc[i, 'stationName'],
        data.loc[i, 'parkingQRBikeCnt'],
        data.loc[i, 'parkingELECBikeCnt']
    )
    popup = folium.Popup(string, max_width=300)
    folium.Marker(
        location=[data.loc[i, 'stationLatitude'], data.loc[i, 'stationLongitude']], 
        popup=popup,
    ).add_to(bike_map)

bike_map.save('./output/bike_map.html')
bike_map